# Findgoal Environment

> A multigrid environment, where the agent has to find a goal in a maze. The agent receives a reward of 1 when it reaches the goal, and 0 otherwise. The episode ends when the agent reaches the goal or after a maximum number of steps.

In [ ]:
#| default_exp envs.findgoal

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import math
from collections import defaultdict
from itertools import repeat


from fastcore.utils import patch
import numpy as np
from numpy.typing import NDArray as ndarray

from typing import Any, Callable, Iterable, Literal, SupportsFloat

import gymnasium as gym
from gymnasium import spaces

from multigrid.envs.base import MultiGridEnv
from multigrid.envs.roomgrid import RoomGrid
from multigrid.utils.random import RandomMixin
from multigrid.core import Grid
from multigrid.core.constants import Direction, Color, Type, TILE_PIXELS
from multigrid.core.world_object import Goal, Marker, Wall, WorldObj
from multigrid.core.agent import Agent, AgentState, NavigationAgent
from multigrid.utils.obs import gen_obs_grid_encoding, gen_obs_grid_image
from multigrid.core.actions import Action, NavigationAction




In [ ]:
#| export
AgentID = int
ObsType = dict[str, Any]


In [ ]:
# #| export
# class FindGoalEnv(RoomGrid):
#     """
#     .. image:: https://i.imgur.com/wY0tT7R.gif
#         :width: 200

#     ***********
#     Description
#     ***********

#     This environment is an empty room, and the goal for each agent is to reach the
#     green goal square, which provides a sparse reward. A small penalty is subtracted
#     for the number of steps to reach the goal.

#     The standard setting is competitive, where agents race to the goal, and
#     only the winner receives a reward.

#     This environment is useful with small rooms, to validate that your RL algorithm
#     works correctly, and with large rooms to experiment with sparse rewards and
#     exploration. The random variants of the environment have the agents starting
#     at a random position for each episode, while the regular variants have the
#     agent always starting in the corner opposite to the goal.

#     *************
#     Mission Space
#     *************

#     "get to the green goal square"

#     *****************
#     Observation Space
#     *****************

#     The multi-agent observation space is a Dict mapping from agent index to
#     corresponding agent observation space.

#     Each agent observation is a dictionary with the following entries:

#     * image : ndarray[int] of shape (view_size, view_size, :attr:`.WorldObj.dim`)
#         Encoding of the agent's partially observable view of the environment,
#         where the object at each grid cell is encoded as a vector:
#         (:class:`.Type`, :class:`.Color`, :class:`.State`)
#     * direction : int
#         Agent's direction (0: right, 1: down, 2: left, 3: up)
#     * mission : Mission
#         Task string corresponding to the current environment configuration

#     ************
#     Action Space
#     ************

#     The multi-agent action space is a Dict mapping from agent index to
#     corresponding agent action space.

#     Agent actions are discrete integer values, given by:

#     +-----+--------------+-----------------------------+
#     | Num | Name         | Action                      |
#     +=====+==============+=============================+
#     | 0   | left         | Turn left                   |
#     +-----+--------------+-----------------------------+
#     | 1   | right        | Turn right                  |
#     +-----+--------------+-----------------------------+
#     | 2   | forward      | Move forward                |
#     +-----+--------------+-----------------------------+
#     | 3   | pickup       | Pick up an object           |
#     +-----+--------------+-----------------------------+
#     | 4   | drop         | Drop an object              |
#     +-----+--------------+-----------------------------+
#     | 5   | toggle       | Toggle / activate an object |
#     +-----+--------------+-----------------------------+
#     | 6   | done         | Done completing task        |
#     +-----+--------------+-----------------------------+

#     *******
#     Rewards
#     *******

#     A reward of ``1 - 0.9 * (step_count / max_steps)`` is given for success,
#     and ``0`` for failure.

#     ***********
#     Termination
#     ***********

#     The episode ends if any one of the following conditions is met:

#     * Any agent reaches the goal
#     * Timeout (see ``max_steps``)

#     *************************
#     Registered Configurations
#     *************************

#     * ``MultiGrid-Empty-5x5-v0``
#     * ``MultiGrid-Empty-Random-5x5-v0``
#     * ``MultiGrid-Empty-6x6-v0``
#     * ``MultiGrid-Empty-Random-6x6-v0``
#     * ``MultiGrid-Empty-8x8-v0``
#     * ``MultiGrid-Empty-16x16-v0``
#     """

#     def __init__(
#         self,
#         size: int = 8,
#         agent_start_pos: tuple[int, int] | None = (1, 1),
#         agent_start_dir: Direction | None = Direction.right,
#         max_steps: int | None = None,
#         joint_reward: bool = False,
#         success_termination_mode: str = 'any',
#         **kwargs):
#         """
#         Parameters
#         ----------
#         size : int, default=8
#             Width and height of the grid
#         agent_start_pos : tuple[int, int], default=(1, 1)
#             Starting position of the agents (random if None)
#         agent_start_dir : Direction, default=Direction.right
#             Starting direction of the agents (random if None)
#         max_steps : int, optional
#             Maximum number of steps per episode
#         joint_reward : bool, default=True
#             Whether all agents receive the reward when the task is completed
#         success_termination_mode : 'any' or 'all', default='any'
#             Whether to terminate the environment when any agent reaches the goal
#             or after all agents reach the goal
#         **kwargs
#             See :attr:`multigrid.base.MultiGridEnv.__init__`
#         """
#         self.agent_start_pos = agent_start_pos
#         self.agent_start_dir = agent_start_dir

#         super().__init__(
#             mission_space="get to the green goal square",
#             grid_size=size,
#             max_steps=max_steps or (4 * size**2),
#             joint_reward=joint_reward,
#             success_termination_mode=success_termination_mode,
#             **kwargs,
#         )

#     def _gen_grid(self, width, height):
#         pass

In [ ]:
# #| export
# from multigrid.envs.roomgrid import RoomGrid
# from multigrid.core.constants import Color, Direction, Type
# from multigrid.core.world_object import WorldObj
# import gymnasium as gym

# class FindGoalEnv(RoomGrid):
#     """
#     Multi-room environment where the agent must find and reach a goal object.
#     The goal is placed in a random room, and the agent must navigate through
#     doors to reach it.
#     """

#     def __init__(
#         self,
#         room_size: int = 7,
#         num_rows: int = 3,
#         num_cols: int = 3,
#         num_agents: int = 1,
#         max_steps: int = 500,
#         **kwargs
#     ):
#         super().__init__(
#             mission_space="find the goal",
#             room_size=room_size,
#             num_rows=num_rows,
#             num_cols=num_cols,
#             agents=num_agents,
#             max_steps=max_steps,
#             **kwargs
#         )

#     # def _gen_grid(self, width: int, height: int):
#     #     # 1. Call parent to build rooms and walls
#     #     super()._gen_grid(width, height)

#     #     # 2. Add doors between ALL rooms so the grid is fully connected
#     #     #    (unlocked so the agent can freely navigate)
#     #     for row in range(self.num_rows):
#     #         for col in range(self.num_cols):
#     #             # Add a right-side door if there's a right neighbor
#     #             if col < self.num_cols - 1:
#     #                 self.add_door(col, row, dir=Direction.right, locked=False)
#     #             # Add a down-side door if there's a bottom neighbor
#     #             if row < self.num_rows - 1:
#     #                 self.add_door(col, row, dir=Direction.down, locked=False)

#     #     # 3. Place goal in a random room
#     #     goal_col = self._rand_int(0, self.num_cols)
#     #     goal_row = self._rand_int(0, self.num_rows)
#     #     goal = WorldObj(type=Type.goal, color=Color.green)
#     #     self.place_in_room(goal_col, goal_row, goal)

#     #     # 4. Place each agent in a random room
#     #     for agent in self.agents:
#     #         self.place_agent(agent)

#     def _gen_grid(self, width, height):
#         super()._gen_grid(width, height)
#         self.connect_all()

#         goal_col = self._rand_int(0, self.num_cols)
#         goal_row = self._rand_int(0, self.num_rows)
#         self.place_in_room(goal_col, goal_row, Goal(color=Color.green))

#         for _ in range(4):
#             col = self._rand_int(0, self.num_cols)
#             row = self._rand_int(0, self.num_rows)
#             self.place_in_room(col, row, Marker(color=self._rand_color()))

#         for agent in self.agents:
#             self.place_agent(agent)


#     def add_distractors(self, col=None, row=None, num_distractors=10, all_unique=True):
#         room_objs = (obj for row_ in self.room_grid for room in row_ for obj in room.objs)
#         room_obj_keys = {(obj.type, obj.color) for obj in room_objs}

#         distractors = []
#         while len(distractors) < num_distractors:
#             color = self._rand_color()
#             type_ = self._rand_elem([Type.key, Type.ball, Type.box])
#             if all_unique and (type_, color) in room_obj_keys:
#                 continue
#             c = col if col is not None else self._rand_int(0, self.num_cols)
#             r = row if row is not None else self._rand_int(0, self.num_rows)
#             distractor, _ = self.add_object(c, r, kind=type_, color=color)
#             room_obj_keys.add((type_, color))
#             distractors.append(distractor)
#         return distractors
    

In [ ]:
# #| export
# @patch
# def _gen_grid(self: EmptyEnv, width, height):
#         """
#         :meta private:
#         """
#         # Create an empty grid
#         self.grid = Grid(width, height)

#         # Generate the surrounding walls
#         self.grid.wall_rect(0, 0, width, height)

#         # Place a goal square in the bottom-right corner
#         self.put_obj(Goal(), width - 2, height - 2)

#         # Place the agent
#         for agent in self.agents:
#             if self.agent_start_pos is not None and self.agent_start_dir is not None:
#                 agent.state.pos = self.agent_start_pos
#                 agent.state.dir = self.agent_start_dir
#             else:
#                 self.place_agent(agent)

In [ ]:
# #| export
# @patch
# def get_goal_state(
#     self: EmptyEnv,
#     agent: Agent,
#     agent_view_size: int,
#     tile_size: int = 32,
#     see_through_walls: bool = False,
# ) -> ndarray:
#     """
#     Returns the goal state RGB image for the given agent.
#     The goal state is being one step away from the goal object,
#     facing it.
#     """
#     # Find the goal object position
#     goal_pos = None
#     for x in range(self.grid.width):
#         for y in range(self.grid.height):
#             obj = self.grid.get(x, y)
#             if isinstance(obj, Goal):
#                 goal_pos = np.array([x, y])
#                 break
#         if goal_pos is not None:
#             break

#     if goal_pos is None:
#         raise ValueError("No goal object found in the grid")

#     # Four possible positions around the goal (right, down, left, up)
#     # and the direction the agent must face to look at the goal
#     candidate_positions = [
#         (goal_pos + np.array([1, 0]),  Direction.left),   # agent to the right, facing left
#         (goal_pos + np.array([-1, 0]), Direction.right),  # agent to the left, facing right
#         (goal_pos + np.array([0, 1]),  Direction.up),     # agent below, facing up
#         (goal_pos + np.array([0, -1]), Direction.down),   # agent above, facing down
#     ]

#     # Pick a valid candidate (inside grid, not a wall)
#     goal_agent_pos = None
#     goal_agent_dir = None
#     for pos, dir in candidate_positions:
#         x, y = pos
#         if 0 <= x < self.grid.width and 0 <= y < self.grid.height:
#             cell = self.grid.get(x, y)
#             if cell is None or isinstance(cell, Goal):
#                 goal_agent_pos = pos
#                 goal_agent_dir = dir
#                 break

#     if goal_agent_pos is None:
#         raise ValueError("No valid position adjacent to goal found")

#     # Create a temporary agent state at the goal position
#     goal_agent = Agent(index=agent.index)
#     goal_agent.state.pos = goal_agent_pos
#     goal_agent.state.dir = goal_agent_dir
#     goal_agent.state.color = agent.state.color.name

#     # Compute obs grid for this goal agent state
#     # We need to temporarily modify agents_states for gen_obs_grid
#     original_pos = agent.state.pos
#     original_dir = agent.state.dir

#     agent.state.pos = goal_agent_pos
#     agent.state.dir = goal_agent_dir

#     # Generate the observation image
#     goal_image = gen_obs_grid_image(
#         self.grid,
#         [goal_agent],
#         self.agent_states,  # uses modified agent state
#         agent_view_size,
#         tile_size=tile_size,
#         see_through_walls=see_through_walls,
#     )[0]

#     # Restore original agent state
#     agent.state.pos = original_pos
#     agent.state.dir = original_dir

#     return goal_image

In [ ]:
# the following creates rooms  divided in between them with walls, and the goal is placed in a random room. Obstacles are scattered in the rooms, and agents are placed randomly as well.
# def _gen_grid(self, width, height):
#     # 1. Create empty grid with surrounding walls
#     self.grid = Grid(width, height)
#     self.grid.wall_rect(0, 0, width, height)

#     # 2. Vertical divider splitting left and right halves
#     mid_x = width // 2
#     gap_y = height // 2

#     for y in range(1, height - 1):
#         if y != gap_y and y != gap_y + 1:
#             self.grid.set(mid_x, y, Wall())

#     # 3. Horizontal divider in the left half only
#     mid_y = height // 2
#     gap_x = width // 4

#     for x in range(1, mid_x):
#         if x != gap_x and x != gap_x + 1:
#             self.grid.set(x, mid_y, Wall())

#     # 4. Scatter rectangular wall obstacles inside rooms
#     rooms = [
#         (1,        1,        mid_x - 1,          mid_y - 1),
#         (1,        mid_y+1,  mid_x - 1,          height - mid_y - 2),
#         (mid_x+1,  1,        width - mid_x - 2,  height - 2),
#     ]

#     obstacles_placed = 0
#     max_attempts = 1000

#     while obstacles_placed < self.num_obstacles and max_attempts > 0:
#         max_attempts -= 1

#         room = rooms[self.np_random.integers(0, len(rooms))]
#         rx, ry, rw, rh = room

#         obs_w = self.np_random.integers(2, min(5, rw - 1))
#         obs_h = self.np_random.integers(1, min(3, rh - 1))

#         ox = self.np_random.integers(rx, rx + rw - obs_w)
#         oy = self.np_random.integers(ry, ry + rh - obs_h)

#         conflict = any(
#             self.grid.get(ox + dx, oy + dy) is not None
#             for dx in range(obs_w)
#             for dy in range(obs_h)
#         )
#         if conflict:
#             continue

#         for dx in range(obs_w):
#             for dy in range(obs_h):
#                 self.grid.set(ox + dx, oy + dy, Wall(color=Color.grey))

#         obstacles_placed += 1

#     # 5. Place goal — fixed corner or random, matching reference logic
#     if getattr(self, 'randomize_goal', True):
#         # Place randomly in the bottom-left room using place_obj
#         goal_pos = self.place_obj(
#             Goal(),
#             top=(1, mid_y + 1),
#             size=(mid_x - 1, height - mid_y - 2),
#             max_tries=100,
#         )
#     else:
#         # Fixed position: bottom-right corner (deterministic)
#         goal_pos = np.asarray([width - 2, height - 2])
#         self.put_obj(Goal(), width - 2, height - 2)

#     # 6. Scatter single-cell wall clutter (matches reference n_clutter)
#     for _ in range(getattr(self, 'n_clutter', 0)):
#         self.place_obj(Wall(), max_tries=100)

#     # 7. Place agents anywhere in the inner grid
#     for agent in self.agents:
#         self.place_agent(
#             agent,
#             top=(1, 1),
#             size=(width - 2, height - 2),
#         )

#     return goal_pos

In [ ]:
#| export
class FindGoalEnv(MultiGridEnv):
    """
    Multi-room environment where agents must find and reach the green goal.
    Rooms are separated by walls with open gaps (no doors).
    Random wall obstacles are scattered inside rooms as visual/navigation clutter.
    """

    def __init__(
        self,
        width: int = 30,
        height: int = 20,
        num_obstacles: int = 25,
        n_clutter: int = 10,
        randomize_goal: bool = True,
        max_steps: int = 500,
        joint_reward: bool = False,
        success_termination_mode: str = 'all',
        allow_agent_overlap: bool = False,
        see_through_walls: bool = True,
        min_goal_spawn_distance: int = 5,
        **kwargs
    ):
        self.num_obstacles = num_obstacles
        self.randomize_goal = randomize_goal
        self.n_clutter = n_clutter
        self.min_goal_spawn_distance = min_goal_spawn_distance
        self._goal_rng = np.random.default_rng()

        super().__init__(
            mission_space="get to the green goal square",
            width=width,
            height=height,
            max_steps=max_steps,
            joint_reward=joint_reward,
            success_termination_mode=success_termination_mode,
            allow_agent_overlap=allow_agent_overlap,
            see_through_walls=see_through_walls,
            **kwargs,
        )

        # reinstantiate the agents to be NavigationAgents (different action space)
        agent_view_size = self.agents[0].view_size if isinstance(self.agents, Iterable) else 7
        self.agent_states = AgentState(self.num_agents) # joint agent state (vectorized)
        self.agents: list[NavigationAgent] = []
        for i in range(self.num_agents):
            agent = NavigationAgent(
                index=i,
                mission_space=self.mission_space,
                view_size=agent_view_size,
                see_through_walls=see_through_walls,
            )
            agent.state = self.agent_states[i]
            self.agents.append(agent)

        # Action enumeration for this environment
        self.actions = NavigationAction

    def _gen_grid(self: MultiGridEnv, width, height):
        # 1. Create empty grid with surrounding walls
        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)

        # 2. Scatter rectangular wall obstacles
        obstacles_placed = 0
        max_attempts = 1000

        while obstacles_placed < self.num_obstacles and max_attempts > 0:
            max_attempts -= 1
            obs_w = self.np_random.integers(2, 5)
            obs_h = self.np_random.integers(1, 3)
            ox = self.np_random.integers(1, width  - obs_w - 1)
            oy = self.np_random.integers(1, height - obs_h - 1)

            conflict = any(
                self.grid.get(ox + dx, oy + dy) is not None
                for dx in range(obs_w)
                for dy in range(obs_h)
            )
            if conflict:
                continue

            for dx in range(obs_w):
                for dy in range(obs_h):
                    self.grid.set(ox + dx, oy + dy, Wall(color=Color.grey))
            obstacles_placed += 1

        # 3. Place goal
        if self.randomize_goal:
            goal_pos = self.place_obj(
                Goal(),
                top=(1, 1),
                size=(width - 2, height - 2),
                max_tries=100,
            )
        else:
            goal_pos = np.asarray([width - 2, height - 2])
            self.put_obj(Goal(), width - 2, height - 2)
        self.goal_pos = goal_pos

        # 4. Extra single-cell clutter
        for _ in range(self.n_clutter):
            self.place_obj(Wall(), max_tries=100)

        # 5. Place agents
        def reject_spawn_fn(env, pos):
            """Reject positions too close to goal (where goal might be visible)"""
            print("Rejecting spawn positions too close to goal")
            print(f"Goal pos: {goal_pos}, Candidate pos: {pos}")
            # dist = abs(pos[0] - goal_pos[0]) + abs(pos[1] - goal_pos[1])
            # another way of doing the same as above is through manhattan distance: dist = np.sum(np.abs(np.array(pos) - goal_pos))
            dist = np.linalg.norm(np.array(pos) - goal_pos, ord=1)
            return dist < self.min_goal_spawn_distance
        
        for agent in self.agents:
            self.place_agent(agent, top=(1, 1), size=(width - 2, height - 2), reject_fn= reject_spawn_fn)

        return goal_pos



In [ ]:
#| export
@patch
def place_agent(
    self: FindGoalEnv,
    agent: NavigationAgent,
    top=None,
    size=None,
    rand_dir=True,
    max_tries=math.inf,
    reject_fn: Callable[[FindGoalEnv, tuple[int, int]], bool] | None = None) -> tuple[int, int]:
    """
    Set agent starting point at an empty position in the grid.
    """
    agent.state.pos = (-1, -1)
    pos = self.place_obj(None, top, size, max_tries=max_tries, reject_fn=reject_fn)
    agent.state.pos = pos

    if rand_dir:
        agent.state.dir = self._rand_int(0, 4)

    return pos

In [ ]:
#| export
@patch
def place_obj(
    self: FindGoalEnv,
    obj: WorldObj | None,
    top: tuple[int, int] = None,
    size: tuple[int, int] = None,
    reject_fn: Callable[[FindGoalEnv, tuple[int, int]], bool] | None = None,
    max_tries=math.inf) -> tuple[int, int]:
    """
    Place an object at an empty position in the grid.

    Parameters
    ----------
    obj: WorldObj
        Object to place in the grid
    top: tuple[int, int]
        Top-left position of the rectangular area where to place the object
    size: tuple[int, int]
        Width and height of the rectangular area where to place the object
    reject_fn: Callable[FindGoalEnv, tuple[int, int]] -> bool
        Function to filter out potential positions
    max_tries: int
        Maximum number of attempts to place the object
    """
    if top is None:
        top = (0, 0)
    else:
        top = (max(top[0], 0), max(top[1], 0))

    if size is None:
        size = (self.grid.width, self.grid.height)

    num_tries = 0

    while True:
        # This is to handle with rare cases where rejection sampling
        # gets stuck in an infinite loop
        if num_tries > max_tries:
            raise RecursionError("rejection sampling failed in place_obj")

        num_tries += 1

        
        if obj is not None and obj.type == Type.goal:
            rng = self._goal_rng
        else:
            rng = self.np_random
            
        pos = (
            rng.integers(top[0], min(top[0] + size[0], self.grid.width)),
            rng.integers(top[1], min(top[1] + size[1], self.grid.height)),
        )

        # Don't place the object on top of another object
        if self.grid.get(*pos) is not None:
            continue

        # Don't place the object where agents are
        if np.bitwise_and.reduce(self.agent_states.pos == pos, axis=1).any():
            continue

        # Check if there is a filtering criterion
        if reject_fn and reject_fn(self, pos):
            continue

        break

    self.grid.set(pos[0], pos[1], obj)

    if obj is not None:
        obj.init_pos = pos
        obj.cur_pos = pos

    return pos



In [ ]:
#| export
@patch
def step(
    self: FindGoalEnv,
    actions: dict[AgentID, NavigationAction]) -> tuple[
        dict[AgentID, ObsType],
        dict[AgentID, SupportsFloat],
        dict[AgentID, bool],
        dict[AgentID, bool],
        dict[AgentID, dict[str, Any]]]:
    """
    Run one timestep of the environment’s dynamics
    using the provided agent actions.

    Parameters
    ----------
    actions : dict[AgentID, NavigationAction]
        Navigation action for each agent acting at this timestep

    Returns
    -------
    observations : dict[AgentID, ObsType]
        Observation for each agent
    rewards : dict[AgentID, SupportsFloat]
        Reward for each agent
    terminations : dict[AgentID, bool]
        Whether the episode has been terminated for each agent (success or failure)
    truncations : dict[AgentID, bool]
        Whether the episode has been truncated for each agent (max steps reached)
    infos : dict[AgentID, dict[str, Any]]
        Additional information for each agent
    """
    self.step_count += 1
    rewards = self.handle_actions(actions)

    # Generate outputs
    observations = self.gen_obs()
    terminations = dict(enumerate(self.agent_states.terminated))
    truncated = self.step_count >= self.max_steps
    truncations = dict(enumerate(repeat(truncated, self.num_agents)))

    # Rendering
    if self.render_mode == 'human':
        self.render()

    return observations, rewards, terminations, truncations, defaultdict(dict)

In [ ]:
#| export
@patch
def handle_actions(
    self: FindGoalEnv, actions: dict[AgentID, NavigationAction]) -> dict[AgentID, SupportsFloat]:
    """
    Handle actions taken by agents.

    Parameters
    ----------
    actions : dict[AgentID, NavigationAction]
        Navigation action for each agent acting at this timestep

    Returns
    -------
    rewards : dict[AgentID, SupportsFloat]
        Reward for each agent
    """
    rewards = {agent_index: 0 for agent_index in range(self.num_agents)}

    # Randomize agent action order
    if self.num_agents == 1:
        order = (0,)
    else:
        order = self.np_random.random(size=self.num_agents).argsort()

    # Update agent states, grid states, and reward from actions
    for i in order:
        if i not in actions:
            continue

        agent, action = self.agents[i], actions[i]

        if agent.state.terminated:
            continue

        # Rotate left
        if action == NavigationAction.left:
            agent.state.dir = (agent.state.dir - 1) % 4

        # Rotate right
        elif action == NavigationAction.right:
            agent.state.dir = (agent.state.dir + 1) % 4

        # Move forward
        elif action == NavigationAction.forward:
            fwd_pos = agent.front_pos
            fwd_obj = self.grid.get(*fwd_pos)

            if fwd_obj is None or fwd_obj.can_overlap():
                if not self.allow_agent_overlap:
                    agent_present = np.bitwise_and.reduce(
                        self.agent_states.pos == fwd_pos, axis=1).any()
                    if agent_present:
                        continue

                agent.state.pos = fwd_pos
                if fwd_obj is not None:
                    if fwd_obj.type == Type.goal:
                        self.on_success(agent, rewards, {})
                    if fwd_obj.type == Type.lava:
                        self.on_failure(agent, rewards, {})

        # Done action (not used by default)
        elif action == NavigationAction.done:
            pass

        else:
            raise ValueError(f"Unknown action: {action}")

    return rewards



In [ ]:
#| export
@patch
def get_goal_state(
    self: FindGoalEnv,
    agent: NavigationAgent,
    agent_view_size: int,
    tile_size: int = 32,
    see_through_walls: bool = False,
) -> ndarray:
    """
    Returns the goal state RGB image for the given agent.
    The goal state is being one step away from the goal object,
    facing it.
    """
    # Find the goal object position
    goal_pos = None
    for x in range(self.grid.width):
        for y in range(self.grid.height):
            obj = self.grid.get(x, y)
            if isinstance(obj, Goal):
                goal_pos = np.array([x, y])
                break
        if goal_pos is not None:
            break

    if goal_pos is None:
        raise ValueError("No goal object found in the grid")

    # Four possible positions around the goal (right, down, left, up)
    # and the direction the agent must face to look at the goal
    candidate_positions = [
        (goal_pos + np.array([1, 0]),  Direction.left),   # agent to the right, facing left
        (goal_pos + np.array([-1, 0]), Direction.right),  # agent to the left, facing right
        (goal_pos + np.array([0, 1]),  Direction.up),     # agent below, facing up
        (goal_pos + np.array([0, -1]), Direction.down),   # agent above, facing down
    ]

    # Pick a valid candidate (inside grid, not a wall)
    goal_agent_pos = None
    goal_agent_dir = None
    for pos, dir in candidate_positions:
        x, y = pos
        if 0 <= x < self.grid.width and 0 <= y < self.grid.height:
            cell = self.grid.get(x, y)
            if cell is None or isinstance(cell, Goal):
                goal_agent_pos = pos
                goal_agent_dir = dir
                break

    if goal_agent_pos is None:
        raise ValueError("No valid position adjacent to goal found")

    # Create a temporary agent state at the goal position
    goal_agent = NavigationAgent(index=agent.index)
    goal_agent.state.pos = goal_agent_pos
    goal_agent.state.dir = goal_agent_dir
    goal_agent.state.color = agent.state.color.name

    # Compute obs grid for this goal agent state
    # We need to temporarily modify agents_states for gen_obs_grid
    original_pos = agent.state.pos
    original_dir = agent.state.dir

    agent.state.pos = goal_agent_pos
    agent.state.dir = goal_agent_dir

    # Generate the observation image
    goal_image = gen_obs_grid_image(
        self.grid,
        [goal_agent],
        self.agent_states,  # uses modified agent state
        agent_view_size,
        tile_size=tile_size,
        see_through_walls=see_through_walls,
    )[0]

    # Restore original agent state
    agent.state.pos = original_pos
    agent.state.dir = original_dir

    return goal_image

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()